# Phase 1 : Setup and Initial Data Exploration

**Project:** *Can You Actually Afford This Loan?* A consumer credit risk model reframed around sustainable affordability instead of pure default prediction.

The primary objective of this notebook is to load the raw data and take a careful look at what is inside. Before building any predictive models, it is essential to understand the structure of the data, check the balance of outcomes, and identify any missing values or severe anomalies. This foundational step ensures that all subsequent cleaning and modelling decisions are built upon sound commercial and statistical assumptions.


In [7]:
# In R you'd write: library(tidyverse)
# In Python each library is imported individually, usually with a short alias
import pandas as pd   # pandas = R's data.frame/tibble world
import numpy as np    # numpy = fast numeric operations that pandas is built on top of

# Make pandas print full-width tables instead of truncating columns with "..."
pd.set_option('display.max_columns', None)


## Loading the Data

The dataset is sourced from the Kaggle **"Give Me Some Credit"** competition and stored in the local data directory. 

The raw file includes an unnamed index column at the beginning. This is explicitly set as the row index during the data load to ensure it is not mistakenly treated as a financial feature.

In [8]:
import os
print("Current working directory:", os.getcwd())  # diagnostic - keep this habit for future path errors

df = pd.read_csv('../data/cs-training.csv', index_col=0)

# .head() shows the first 5 rows - same idea as head(df) in R
df.head()


Current working directory: c:\Users\imraa\OneDrive\Desktop\credit-risk-affordability-starter\credit-risk-affordability\notebooks


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


## Size and Data Types

The overall dimensions of the dataset and the data types for each column are checked below. This confirms the total number of records and provides an initial summary of the available data and missing values across all features.


In [9]:
print(df.shape)
df.info()


(150000, 11)
<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 1 to 150000
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      150000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 2   age                                   150000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 4   DebtRatio                             150000 non-null  float64
 5   MonthlyIncome                         120269 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 7   NumberOfTimes90DaysLate               150000 non-null  int64  
 8   NumberRealEstateLoansOrLines          150000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 10  NumberOfDependents                    146076 non-null  float64
dty

## Target Variable Distribution

The target variable for this project is `SeriousDlqin2yrs`. A value of `1` indicates that the borrower was 90 or more days past due on a payment within a two year window, while `0` indicates they were not.

It is crucial to check the proportion of these outcomes early on. In consumer lending, defaults are naturally rare events, which creates a significant class imbalance in the data. Because of this imbalance, relying on standard accuracy as a performance metric would be misleading. A model could simply predict zero defaults and still appear highly accurate while failing to catch any real risk.

In [10]:
df['SeriousDlqin2yrs'].value_counts(normalize=True)


SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64

## Checking for Missing Data

The dataset is evaluated to identify any missing values. Fields such as `MonthlyIncome` and `NumberOfDependents` show significant gaps. Real world income data is frequently incomplete, and deciding how to address this missing information is a crucial analytical choice. It requires a thoughtful approach in the next phase rather than a quick default fix.


In [11]:
df.isnull().sum()


SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

## Summary Statistics and Anomaly Detection

Generating summary statistics provides the minimum, maximum, mean, and quartiles for all numeric columns. A review of these metrics reveals several impossible entries in the dataset. For instance, the maximum values for `RevolvingUtilizationOfUnsecuredLines` and `DebtRatio` reach into the thousands, whereas these financial ratios should typically be much lower. Similarly, the minimum value for `age` is zero. Real consumer credit datasets frequently contain these types of data entry errors. Identifying and addressing these anomalies at this stage is essential for building a robust and reliable assessment tool.

In [12]:
df.describe()


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.029867,41.000000,0.000000,0.175074,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,0.000000,0.559046,63.000000,0.000000,0.868254,8.249000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


## Key Observations and Next Steps

The initial exploration reveals several areas that require attention before modelling:

* **Class Imbalance:** A small proportion of the dataset represents actual defaults, establishing the need for appropriate evaluation metrics rather than basic accuracy.
* **Missing Data:** Significant gaps exist in the income and dependents fields.
* **Data Anomalies:** There are physically impossible values in the age and financial ratio columns.

In the next phase, these data quality issues will be addressed directly. The focus will then shift to feature engineering, specifically translating abstract financial ratios into concrete affordability metrics.